# 使用zfs

## 创建磁盘

In [8]:
!sudo losetup

In [9]:
!sudo zpool status

no pools available


In [10]:
# 假设有4个盘构建raid5
!fallocate -l 1G ./disk1.img
!fallocate -l 1G ./disk2.img
!fallocate -l 3G ./disk3.img
!fallocate -l 4G ./disk4.img

In [11]:
import os
for i in range(1, 5):
    os.environ["i"] = str(i)
    !sudo losetup -f ./disk$i.img

In [12]:
!sudo losetup

NAME       SIZELIMIT OFFSET AUTOCLEAR RO BACK-FILE                   DIO LOG-SEC
/dev/loop1         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk2.img
                                                                       0     512
/dev/loop2         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk3.img
                                                                       0     512
/dev/loop0         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk1.img
                                                                       0     512
/dev/loop3         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk4.img
                                                                       0     512


In [13]:
!sudo zpool create -f testpool raidz1 /dev/loop0 /dev/loop1 /dev/loop2 /dev/loop3

In [15]:
!sudo zpool status

  pool: testpool
 state: ONLINE
config:

	NAME        STATE     READ WRITE CKSUM
	testpool    ONLINE       0     0     0
	  raidz1-0  ONLINE       0     0     0
	    loop0   ONLINE       0     0     0
	    loop1   ONLINE       0     0     0
	    loop2   ONLINE       0     0     0
	    loop3   ONLINE       0     0     0

errors: No known data errors


In [16]:
!sudo zpool list

NAME       SIZE  ALLOC   FREE  CKPOINT  EXPANDSZ   FRAG    CAP  DEDUP    HEALTH  ALTROOT
testpool  3.75G   191K  3.75G        -         -     1%     0%  1.00x    ONLINE  -


In [17]:
!df /testpool -h

Filesystem      Size  Used Avail Use% Mounted on
testpool        2.7G  128K  2.7G   1% /testpool


## 添加硬盘

In [18]:
!fallocate -l 2G ./disk5.img

In [20]:
!sudo losetup -f ./disk5.img

In [22]:
!losetup | grep disk5

/dev/loop4         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk5.img   0     512


In [24]:
!sudo zpool status

  pool: testpool
 state: ONLINE
config:

	NAME        STATE     READ WRITE CKSUM
	testpool    ONLINE       0     0     0
	  raidz1-0  ONLINE       0     0     0
	    loop0   ONLINE       0     0     0
	    loop1   ONLINE       0     0     0
	    loop2   ONLINE       0     0     0
	    loop3   ONLINE       0     0     0

errors: No known data errors


In [26]:
!sudo zpool attach testpool raidz1-0 /dev/loop4

In [27]:
!sudo zpool list

NAME       SIZE  ALLOC   FREE  CKPOINT  EXPANDSZ   FRAG    CAP  DEDUP    HEALTH  ALTROOT
testpool  4.75G   491K  4.75G        -         -     2%     0%  1.00x    ONLINE  -


In [28]:
!df /testpool -h

Filesystem      Size  Used Avail Use% Mounted on
testpool        3.5G  128K  3.5G   1% /testpool


In [30]:
# 拷贝文件后
!df /testpool -h

Filesystem      Size  Used Avail Use% Mounted on
testpool        3.5G  2.8G  720M  80% /testpool


## 扩容硬盘
最后10+3G的硬盘变成了7.1G

In [32]:
# 某个硬盘坏
!sudo zpool offline testpool /dev/loop0

In [33]:
# 创建新的景象
!fallocate -l 2G ./disk6.img

In [34]:
!sudo losetup -f ./disk6.img

In [36]:
!sudo losetup -l

NAME       SIZELIMIT OFFSET AUTOCLEAR RO BACK-FILE                   DIO LOG-SEC
/dev/loop1         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk2.img
                                                                       0     512
/dev/loop4         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk5.img
                                                                       0     512
/dev/loop2         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk3.img
                                                                       0     512
/dev/loop0         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk6.img
                                                                       0     512
/dev/loop3         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk4.img
          

In [38]:
!sudo zpool status

  pool: testpool
 state: DEGRADED
status: One or more devices has been taken offline by the administrator.
	Sufficient replicas exist for the pool to continue functioning in a
	degraded state.
action: Online the device using 'zpool online' or replace the device with
	'zpool replace'.
  scan: scrub repaired 0B in 00:00:01 with 0 errors on Sun Mar 15 14:23:33 2026
expand: expanded raidz1-0 copied 294K in 00:00:01, on Sun Mar 15 11:08:44 2026
config:

	NAME        STATE     READ WRITE CKSUM
	testpool    DEGRADED     0     0     0
	  raidz1-0  DEGRADED     0     0     0
	    loop0   OFFLINE      0     0     0
	    loop1   ONLINE       0     0     0
	    loop2   ONLINE       0     0     0
	    loop3   ONLINE       0     0     0
	    loop4   ONLINE       0     0     0

errors: No known data errors


In [ ]:
!sudo zpool replace testpool /dev/loop0 /loop0

In [40]:
!fallocate -l 2G disk7.img

In [42]:
!sudo losetup -f ./disk7.img

In [43]:
!sudo losetup -l

NAME       SIZELIMIT OFFSET AUTOCLEAR RO BACK-FILE                   DIO LOG-SEC
/dev/loop1         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk2.img
                                                                       0     512
/dev/loop4         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk5.img
                                                                       0     512
/dev/loop2         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk3.img
                                                                       0     512
/dev/loop0         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk6.img
                                                                       0     512
/dev/loop5         0      0         0  0 /home/wangx/github/linux-reference/software/zfs/测试不同容量的磁盘能否做raid5/disk7.img
          

In [44]:
!sudo zpool status

  pool: testpool
 state: DEGRADED
status: One or more devices has been taken offline by the administrator.
	Sufficient replicas exist for the pool to continue functioning in a
	degraded state.
action: Online the device using 'zpool online' or replace the device with
	'zpool replace'.
  scan: resilvered 747M in 00:00:00 with 0 errors on Sun Mar 15 14:28:47 2026
expand: expanded raidz1-0 copied 294K in 00:00:01, on Sun Mar 15 11:08:44 2026
config:

	NAME        STATE     READ WRITE CKSUM
	testpool    DEGRADED     0     0     0
	  raidz1-0  DEGRADED     0     0     0
	    loop0   ONLINE       0     0     0
	    loop1   OFFLINE      0     0     0
	    loop2   ONLINE       0     0     0
	    loop3   ONLINE       0     0     0
	    loop4   ONLINE       0     0     0

errors: No known data errors


In [46]:
!sudo zpool replace  testpool /dev/loop1 /dev/loop5

In [49]:
!df -h /testpool

Filesystem      Size  Used Avail Use% Mounted on
testpool        3.5G  2.8G  720M  80% /testpool


In [53]:
!sudo zpool status

  pool: testpool
 state: ONLINE
  scan: resilvered 747M in 00:00:01 with 0 errors on Sun Mar 15 14:32:15 2026
expand: expanded raidz1-0 copied 294K in 00:00:01, on Sun Mar 15 11:08:44 2026
config:

	NAME        STATE     READ WRITE CKSUM
	testpool    ONLINE       0     0     0
	  raidz1-0  ONLINE       0     0     0
	    loop0   ONLINE       0     0     0
	    loop5   ONLINE       0     0     0
	    loop2   ONLINE       0     0     0
	    loop3   ONLINE       0     0     0
	    loop4   ONLINE       0     0     0

errors: No known data errors


In [54]:
!sudo zpool online -e testpool /dev/loop0 /dev/loop2 /dev/loop3 /dev/loop4 /dev/loop5

In [55]:
!df -h /testpool

Filesystem      Size  Used Avail Use% Mounted on
testpool        7.1G  2.8G  4.4G  39% /testpool
